# 02 · Fine-tune (QLoRA)

**Phase notebook** — feature `004-notebook-split`. Produces `lora_adapter/` on Google Drive.
Run top-to-bottom on a fresh Colab runtime. All constants come from the shared `config.py`.

## Deviations
- **Notebook decomposition (constitution v1.2.0 §I / §Notebook Decomposition).** One of four phase
  notebooks split from the original single `notebook.ipynb`, for modularity/reuse and dependency-conflict
  isolation. Phases hand off **only** via Drive artifacts; the single shared `config.py` is the only
  cross-notebook import. This supersedes the single-notebook risk-first stage order (recorded per §Governance).


In [ ]:
# ── Cell 1 · Bootstrap: fetch shared config.py, install THIS phase's deps ──
# The four phase notebooks share exactly ONE module (config.py), fetched from the
# repo (constitution v1.2.0 §Notebook Decomposition). Set REPO_URL to your repo.
# For a PRIVATE repo, use a token URL (https://<TOKEN>@github.com/...) or the
# raw-download fallback shown below.
import os, sys, subprocess

REPO_URL = "https://github.com/YOUR_ORG/fine-tuning-demo.git"  # TODO: set to your repo
REPO_DIR = "/content/fine-tuning-demo"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
sys.path.insert(0, REPO_DIR)
# Fallback (config.py only, e.g. private repo without clone access):
#   from urllib.request import urlretrieve
#   urlretrieve("<raw-url>/config.py", "/content/config.py"); sys.path.insert(0, "/content")

import config  # the ONLY cross-notebook import (FR-012)

# Fine-tune phase: training stack ONLY (no RAG/UI stack; vLLM excluded).
# This phase's MINIMAL dependency subset (§III / FR-004) — pinned HERE, not shared.
PINNED_REQS = [
    "unsloth",
    "unsloth_zoo",
    "transformers>=5.2",
    "trl>=0.15",
    "peft>=0.14",
    "bitsandbytes>=0.45",
    "datasets>=3.0",
]

config.mount_drive()
config.install_deps(PINNED_REQS)
config.set_seeds()

In [ ]:
# ── Cell 2 · QLoRA fine-tune (ported from monolith Stages 2 + 5) ──
import os, json, shutil, torch
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

# Resolve the GPU profile (A100 default / T4 fallback) BEFORE loading the model (\u00a7III)
config.select_profile(config.detect_vram_gb())

ADAPTER_PATH = config.lora_adapter_path()
_gate = config.artifact_gate("LoRA adapter", ADAPTER_PATH, "adapter_model.safetensors", rebuild_verb="Retrain")
if _gate == "skip":
    print("\u23e9 Skipping training \u2014 lora_adapter already on Drive")
else:
    # Validate JSONL (ported from Stage 5)
    assert os.path.exists(config.DATASET_PATH), f"Dataset not found: {config.DATASET_PATH}"
    valid, errors = [], []
    with open(config.DATASET_PATH, encoding="utf-8") as f:
        for i, line in enumerate(f, 1):
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            try:
                obj  = json.loads(line)
                msgs = obj.get("messages", [])
                assert len(msgs) >= 2,                 "Need >=2 messages"
                assert msgs[0]["role"] == "user",      "First role must be user"
                assert msgs[0]["content"].strip(),     "User content is empty"
                assert msgs[1]["role"] == "assistant", "Second role must be assistant"
                assert msgs[1]["content"].strip(),     "Assistant content is empty"
                valid.append(obj)
            except Exception as e:
                errors.append(f"  Line {i}: {e}")
    if errors:
        raise ValueError("JSONL validation failed:\n" + "\n".join(errors[:10]))
    if len(valid) < 200:
        raise ValueError(f"Need >=200 training pairs, got {len(valid)}")
    if len(valid) < 500:
        print(f"\u26a0\ufe0f  {len(valid)} pairs (target 500+) \u2014 more data may improve results")
    print(f"\u2705 JSONL valid: {len(valid)} pairs")

    os.makedirs(config.DRIVE_BASE, exist_ok=True)
    shutil.copy(config.DATASET_PATH, f"{config.DRIVE_BASE}/training_dataset.jsonl")

    # Load base model + apply LoRA
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=config.MODEL_ID, max_seq_length=config.MAX_SEQ_LEN,
        load_in_4bit=True, dtype=None)
    model = FastLanguageModel.get_peft_model(
        model, r=config.LORA_R, lora_alpha=config.LORA_ALPHA,
        target_modules=config.LORA_TARGETS, lora_dropout=config.LORA_DROPOUT,
        bias="none", use_gradient_checkpointing="unsloth")

    # Format + train
    dataset = load_dataset("json", data_files=config.DATASET_PATH, split="train")
    dataset = dataset.map(lambda ex: {"text": tokenizer.apply_chat_template(
        ex["messages"], tokenize=False, add_generation_prompt=False)})
    trainer = SFTTrainer(
        model=model, tokenizer=tokenizer, train_dataset=dataset,
        dataset_text_field="text", max_seq_length=config.MAX_SEQ_LEN,
        args=TrainingArguments(
            per_device_train_batch_size=config.TRAIN_BATCH,
            gradient_accumulation_steps=config.GRAD_ACCUM,
            num_train_epochs=config.TRAIN_EPOCHS, learning_rate=config.LEARNING_RATE,
            fp16=not torch.cuda.is_bf16_supported(), bf16=torch.cuda.is_bf16_supported(),
            logging_steps=10, output_dir="outputs", lr_scheduler_type="cosine", seed=config.SEED))
    stats = trainer.train()
    print(f"Training complete \u2014 steps: {stats.global_step}, final loss: {stats.training_loss:.4f}")

    # Save adapter + tokenizer, then stamp the drift sidecar (FR-013) LAST
    model.save_pretrained(ADAPTER_PATH)
    tokenizer.save_pretrained(ADAPTER_PATH)
    config.write_meta(ADAPTER_PATH, {
        "base_model_id": config.MODEL_ID, "max_seq_len": config.MAX_SEQ_LEN,
        "lora_r": config.LORA_R, "lora_alpha": config.LORA_ALPHA,
        "lora_targets": config.LORA_TARGETS, "seed": config.SEED})
    print(f"\u2705 Adapter saved to Drive: {ADAPTER_PATH}")